# 第 6 章 商品关联规则与组合销售

本章商业问题：**哪些商品经常一起买，如何转成推荐、套餐和凑单策略？**

本章所有代码默认优先读取真实 ETL 接口 `http://192.168.31.47:38173/api/etl`。如果接口暂时不可用，会自动回退到本地 SQLite 后备数据，保证课堂可以继续进行。

## 0. 先建立商业问题意识

在商业课里，代码不是第一步。第一步是判断：这个问题影响收入、成本、用户体验、库存风险，还是营销效率？只有先说清楚商业目标，后面的数据选择和模型选择才不会跑偏。

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if COURSE_ROOT.name in ["notebooks", "student_notebooks", "teacher_notebooks"]:
    COURSE_ROOT = COURSE_ROOT.parent
elif not (COURSE_ROOT / "course_utils").exists():
    COURSE_ROOT = Path("..").resolve()

sys.path.insert(0, str(COURSE_ROOT))

from course_utils.data_loader import (
    API_BASE, load_table, get_metrics, get_quality_report,
    get_table_catalog, get_schema, paid_orders, api_status, query_table
)
from course_utils.business import money, pct, section

try:
    import matplotlib.pyplot as plt
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

print("课程目录:", COURSE_ROOT)
print("ETL API:", API_BASE)
print("API 状态:", api_status())

## 1. 查看 ETL 数据资产

下面先查看 ETL 接口暴露了哪些表。请注意 `dim_` 开头的是维度表，通常描述对象；`fact_` 开头的是事实表，通常记录业务事件。

In [ ]:
catalog = get_table_catalog()
tables = catalog["tables"]
print("可用表数量:", catalog.get("total", len(tables)))
for t in tables[:12]:
    print(t["tableName"], t.get("recordCount"), t.get("type"), t.get("description", ""))

In [ ]:
import pandas as pd
products = load_table("dim_product", limit=100000)[["sku_id", "sku_name", "category_name", "price", "cost"]]
top_sku = [
    "sku_00001", "sku_00002", "sku_00005", "sku_00003",
    "sku_00008", "sku_00009", "sku_00022", "sku_00023",
    "sku_00153", "sku_00183", "sku_00098", "sku_00073",
    "sku_00232", "sku_00242", "sku_00828", "sku_00803",
    "sku_00292", "sku_00294",
]
frames = []
for sku_id in top_sku:
    frames.append(query_table("fact_order_item", limit=5000, sku_id=sku_id))
items = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["order_item_id"])
basket = items.groupby("order_id")["sku_id"].apply(set)
rules = []
for a in top_sku:
    has_a = basket.apply(lambda s: a in s)
    support_a = has_a.mean()
    for b in top_sku:
        if a == b:
            continue
        has_b = basket.apply(lambda s: b in s)
        both = (has_a & has_b).mean()
        if support_a > 0 and has_b.mean() > 0:
            confidence = both / support_a
            lift = confidence / has_b.mean()
            if both >= 0.001 and confidence >= 0.03 and lift > 1.05:
                rules.append([a, b, both, confidence, lift])
rules_df = pd.DataFrame(rules, columns=["antecedent", "consequent", "support", "confidence", "lift"]).sort_values(["lift", "confidence"], ascending=False)
rules_df.head(10)

## 商业解释

支持度说明组合出现得够不够多，置信度说明买了 A 后买 B 的概率，提升度说明这个组合是不是比随机更强。真正上线推荐前，还要检查库存、毛利、价格带和是否过度打扰用户。

In [ ]:
assert len(rules_df) >= 0
print("第 06 章验证通过")